In [31]:
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
import json
from pathlib import Path

def split_xy(df, target):
    y = df[target]
    X = df.drop(columns=[target])
    return X, y

class MissingValueHandler(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = X.copy()
        self.int_cols_ = X.select_dtypes(include=[np.integer]).columns.tolist()
        self.float_cols_ = X.select_dtypes(include=[np.floating]).columns.tolist()
        self.cat_cols_ = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()
        self.medians_ = X[self.int_cols_].median(numeric_only=True)
        self.means_ = X[self.float_cols_].mean(numeric_only=True)
        return self
    def transform(self, X):
        X = X.copy()
        if self.int_cols_:   
            X[self.int_cols_] = X[self.int_cols_].fillna(self.medians_)
        if self.float_cols_: 
            X[self.float_cols_] = X[self.float_cols_].fillna(self.means_)
        if self.cat_cols_:   
            X[self.cat_cols_] = X[self.cat_cols_].fillna("(NA)").astype("string")
        return X

def build_two_stage_preprocessor():
    stage1 = MissingValueHandler()
    enc_scale = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), make_column_selector(dtype_include=[np.number])),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True),
                    make_column_selector(dtype_include=["object","string","category"])),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )
    return Pipeline([("stage1_missing", stage1), ("stage2_encode_scale", enc_scale)])

def make_folds(y, n_splits=5, seed=42, out=None, name="dataset"):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    folds = [{"train_idx": tr.tolist(), "valid_idx": va.tolist()} for tr,va in skf.split(np.zeros(len(y)), y)]
    if out:
        Path(out).mkdir(parents=True, exist_ok=True)
        Path(out, f"{name}_skf{n_splits}.json").write_text(json.dumps(folds))
    return folds



In [32]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("data/raw")
AVAILABLE = {
    "car": {"file": "car.csv", "target": "class"},
    "aps": {"file": "aps_training.csv", "target": "class", "na_values": ["na"]},
    "covertype": {"file": "covertype.csv", "target": "Cover_Type"},
    "jannis": {"file": "jannis.csv", "target": "__target__"},
}

def _exists(name): 
    return (DATA_DIR / AVAILABLE[name]["file"]).exists()

print("Available files:")
for k in AVAILABLE: 
    print(f" - {k:10s} -> {AVAILABLE[k]['file']}  {'✓' if _exists(k) else '✗'}")

Available files:
 - car        -> car.csv  ✓
 - aps        -> aps_training.csv  ✓
 - covertype  -> covertype.csv  ✓
 - jannis     -> jannis.csv  ✓


In [33]:
DATASET = "jannis"

In [34]:
from IPython.display import display
import pandas as pd

name = DATASET
info = AVAILABLE[name]
path = (DATA_DIR / info["file"])
read_kwargs = {}
print(info)
if "na_values" in info: 
    read_kwargs["na_values"] = info["na_values"]

df = pd.read_csv(path, **read_kwargs)
print(f"Loaded: {name} -> {path.name}")
print("Shape:", df.shape)
print("Memory MB (approx):", round(df.memory_usage(index=True, deep=True).sum()/1e6, 2))
print("Columns sample:", list(df.columns)[:12], "..." if df.shape[1] > 12 else "")
display(df.head())

{'file': 'jannis.csv', 'target': '__target__'}
Loaded: jannis -> jannis.csv
Shape: (83733, 55)
Memory MB (approx): 36.84
Columns sample: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12'] ...


,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V46,V47,V48,V49,V50,V51,V52,V53,V54,__target__
0,0.047095,0.316667,0.288889,0.180925,0.101677,0.252014,0.235673,0.054190,0.485197,253.6360,...,0.421830,0.401547,0.036195,153.2040,-7.584500,21.9080,13.0200,0.086289,1.074020,3
1,0.464873,1.000000,0.483333,0.502843,0.233860,0.998355,0.465075,0.016096,0.038194,141.3640,...,0.356803,0.439840,0.022729,123.5940,0.813364,19.8483,18.3443,0.039786,0.333176,3
2,0.053872,0.516667,0.180556,0.431467,0.064608,0.412341,0.151937,0.056719,0.422519,238.9570,...,0.583339,0.611528,0.042459,147.7000,-0.428918,35.7166,15.8205,0.164682,1.175180,3
3,0.030475,0.245833,0.175000,0.128515,0.438525,0.207337,0.146200,0.049183,0.291633,52.4554,...,0.553694,0.417488,0.059169,147.5880,0.108469,13.0906,21.0650,0.028791,1.215070,1
4,0.038883,0.256250,0.225000,0.128165,0.618360,0.252958,0.171803,0.059235,0.325605,60.1405,...,0.453029,0.687470,0.051630,85.8803,-1.058330,23.8815,10.5757,0.005814,0.222675,1


In [35]:
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
import numpy as np

target_col = info["target"]
X, y = split_xy(df, target_col)

pre = build_two_stage_preprocessor()

n_classes = y.nunique()
if n_classes > 2:
    objective = "multi:softprob"
    eval_metric = "mlogloss"
    extra = {"num_class": int(n_classes)}
else:
    objective = "binary:logistic"
    eval_metric = "logloss"
    extra = {}

xgb_params = dict(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_lambda=1.0,
    objective=objective,
    eval_metric=eval_metric,
    random_state=42,
    tree_method="hist",
    **extra
)

use_gpu = False
try:
    # Prefer modern API
    clf = XGBClassifier(**xgb_params, device="cuda")
    use_gpu = True
except TypeError:
    # Older XGB: use gpu_hist if available
    clf = XGBClassifier(**{**xgb_params, "tree_method": "gpu_hist"})
    use_gpu = True
except Exception:
    # Truly CPU-only fallback
    clf = XGBClassifier(**xgb_params)

pipe = Pipeline([("pre", pre), ("clf", clf)])

print(f"XGBoost device: {'GPU' if use_gpu else 'CPU'}")

folds = make_folds(y, n_splits=4, seed=42, out="data/splits", name=name)

idx = X.sample(min(5000, len(X)), random_state=0).index
_ = pipe.fit(X.loc[idx], y.loc[idx])
print("Pipeline smoke-fit OK.")

XGBoost device: GPU
Pipeline smoke-fit OK.


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, log_loss
import numpy as np

scores = []
for i, f in enumerate(folds, 1):
    tr, va = f["train_idx"], f["valid_idx"]
    pipe.fit(X.iloc[tr], y.iloc[tr])
    proba = pipe.predict_proba(X.iloc[va])
    pred  = proba.argmax(1) if proba.shape[1] > 1 else (proba[:,0] >= 0.5).astype(int)
    yv = y.iloc[va]
    acc = accuracy_score(yv, pred)
    f1m = f1_score(yv, pred, average="macro")
    ll  = log_loss(yv, proba, labels=np.unique(y))
    scores.append({"fold": i, "accuracy": acc, "f1_macro": f1m, "logloss": ll})
scores

In [ ]:
'''
from joblib import dump
pipe.fit(X, y)
Path("models").mkdir(exist_ok=True)
dump(pipe, f"models/{name}_xgb_pipeline.joblib")
print("Saved:", f"models/{name}_xgb_pipeline.joblib")
'''